# Oil 1-Min Predictor — Exploratory Analysis

This notebook is for ad-hoc exploration after running the pipeline.
Run `scripts/run_pipeline.py` first to generate `data/artifacts/oos_predictions.parquet`.

In [ ]:
import sys
sys.path.insert(0, '..')  # so we can import src

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from src.utils import load_config, resolve_path
from src.data_loader import load_and_clean
from src.feature_engineering import build_features, get_feature_names
from src.metrics import compute_metrics
from src.backtest import run_backtest, compute_backtest_stats, threshold_sweep

cfg = load_config('../config/config.yaml')

## 1. Load OOS predictions

In [ ]:
oos = pd.read_parquet('../data/artifacts/oos_predictions.parquet')
print(f"OOS shape: {oos.shape}")
print(f"Folds: {oos['fold'].nunique()}")
oos.head()

## 2. Quick metrics

In [ ]:
from src.metrics import compute_metrics
m = compute_metrics(oos['y_true'].values, oos['y_pred'].values)
for k, v in m.items():
    print(f"  {k:<18}: {v:.6f}")

## 3. Backtest + threshold sweep

In [ ]:
sweep = threshold_sweep(
    oos,
    thresholds=cfg['backtest']['threshold_sweep'],
    cost_per_trade=cfg['backtest']['cost_per_trade'],
    slippage=cfg['backtest']['slippage'],
    annualise_factor=cfg['backtest']['annualise_factor'],
)
sweep[['annualised_sharpe', 'total_log_return', 'max_drawdown', 'hit_rate', 'n_trades']]

## 4. Prediction distribution by hour

In [ ]:
oos['hour'] = oos.index.hour
oos.groupby('hour')['y_pred'].agg(['mean', 'std', 'count']).plot(
    subplots=True, figsize=(12, 6), title='Prediction statistics by hour'
)
plt.tight_layout()

## 5. Correlation by hour (signal quality analysis)

In [ ]:
corr_by_hour = oos.groupby('hour').apply(
    lambda g: np.corrcoef(g['y_true'], g['y_pred'])[0, 1]
)
corr_by_hour.plot.bar(figsize=(10, 3), title='Pearson Corr by Hour of Day')
plt.axhline(0, color='k', lw=0.5)
plt.ylabel('Correlation')
plt.tight_layout()

## 6. Feature importance (from last-fold model)

In [ ]:
import pickle, json

with open('../data/artifacts/model_last_fold.pkl', 'rb') as f:
    model = pickle.load(f)
with open('../data/artifacts/feature_names.json') as f:
    feature_names = json.load(f)

importance = pd.Series(model.feature_importances_, index=feature_names)
importance.sort_values(ascending=True).tail(25).plot.barh(figsize=(8, 7))
plt.title('Top-25 Feature Importances (last fold)')
plt.tight_layout()

## 7. Rolling Sharpe (60-day window)

In [ ]:
bt = run_backtest(
    oos,
    threshold=cfg['backtest']['threshold'],
    cost_per_trade=cfg['backtest']['cost_per_trade'],
    slippage=cfg['backtest']['slippage'],
)

window = 390 * 60  # ~60 trading days in minutes
roll_sharpe = (
    bt['strategy_ret'].rolling(window).mean() /
    bt['strategy_ret'].rolling(window).std()
) * np.sqrt(cfg['backtest']['annualise_factor'])

roll_sharpe.plot(figsize=(12, 3), title='Rolling Sharpe (60-day window)')
plt.axhline(0, color='k', lw=0.5)
plt.tight_layout()